# Inspección Inicial del Dataset
## Proyecto Integrador — Minería de Datos I

---

### Objetivo de este notebook

Este notebook tiene como único propósito **observar y documentar** el estado inicial del dataset provisto por el profe.

La información que generemos de aquí servirá como evidencia para poder justificar porque hice ciertas cosas en la sección 02: "**Calidad, Limpieza y Preparación de Datos**"

### Dataset analizado

- **Archivo original:** "data/raw/streaming_users_dirty.json"
- **Descripción:** registro de usuarios de una plataforma de streaming con información sobre su comportamiento, plan de suscripción y características demográficas.

Se importa la librería para hacer el análisis y se llama al dataset que se va a trabajar

In [ ]:
import pandas as pd

df = pd.read_json('../data/raw/streaming_users_dirty.json')


In [59]:
print(f"Tipos de datos: {df.info()}")

<class 'pandas.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   str    
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   str    
 5   favorite_genre            7920 non-null   str    
 6   last_login_date           7840 non-null   str    
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 510.1 KB
Tipos de datos: None


In [ ]:
print(f"Cantidad de filas x columnas: {df.shape}")
print("="*39)
print(f"Lista de nulos:\n\n{df.isnull().sum()}")
print("="*39)
print(f"Primeras 5 filas del dataset:\n\n{df.head(5)}")


Cantidad de filas x columnas: (8160, 8)
Lista de nulos:

user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins     193
country                       0
favorite_genre              240
last_login_date             320
customer_support_tickets      0
dtype: int64
Primeras 5 filas del dataset:

   user_id  age subscription_plan  monthly_watch_time_mins   country  \
0    10000   39          Estándar                    805.8    Brasil   
1    10001   37          Estándar                   1173.4  Colombia   
2    10002   28            Básico                    401.0  Colombia   
3    10003   43            Básico                     62.4   Uruguay   
4    10004   51            Básico                    477.8      Perú   

  favorite_genre last_login_date  customer_support_tickets  
0          Crime      2025-03-04                        99  
1          Crime      2019-04-02                         2  
2          Crime      2018-

**Inconsistencias**

con describe() se puede encontrar ciertas inconsistencias en nuestros datos numéricos, podemos verlo desde distintos puntos de vista:
- **age** tiene usuarios con edades imposibles, algunos **negativos** como -5, otros directamente demasiado altos como 150, en general es una columna que tendrá que ser modificada de dos maneras, una es eliminando primero las negativas y otra es que determinemos un rango de edad (se hará en una sección posterior a esta).
- **monthly_watch_time_mins** tiene valores "faltantes" (en realidad aparecen como NaN) y se repite los valores negativos como lo tenía la columna **age**, deberán ser eliminados. también buscar primero que puede hacerse con nuestros valores imposibles teniendo en cuenta cuanto puede llegar a consumir una persona por mes.
- **customer_support_tickets** tiene valores negativos (también a solucionar como las features anteriores), y valores muy extensos como la columna anterior que tienen que ser corregidos, estos valores requerirán revisión en la etapa de limpieza.

In [32]:
df.describe().round(2)

,user_id,age,monthly_watch_time_mins,customer_support_tickets
count,8160.00,8160.00,7967.00,8160.00
mean,13995.43,34.10,1107.35,1.80
std,2310.81,14.51,5310.44,11.33
min,10000.00,-5.00,-120.00,-1.00
25%,11987.75,25.00,489.20,0.00
50%,13998.50,33.00,757.40,1.00
75%,15997.25,42.00,1045.70,1.00
max,17999.00,150.00,99999.00,150.00


Para la verificación de los valores alfanuméricos, se saca la lista de valores de algunos de ellos para verificar cualquier error de tipeo, observando un patrón recurrente de errores que va desde el uso de inglés para referirse a ciertos valores que fueron escritos en español, hasta también faltas de acentuación, palabras enteramente escritas en mayúsculas o minúsculas y abreviaciones

In [55]:
df['subscription_plan'].value_counts()

subscription_plan
Básico       3450
Estándar     2711
Premium      1519
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22
Name: count, dtype: int64

In [51]:
df['country'].value_counts()

country
Brasil        1132
Chile         1132
México        1129
Uruguay       1124
Perú          1120
Colombia      1116
Argentina     1087
colombia        27
méxico          25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
MEX             16
Chile           16
argentina       16
PER             16
chile           15
Mexico          15
Peru            15
BRA             15
brasil          13
perú            12
ARG             10
Argentina       10
Name: count, dtype: int64


In [54]:
df['favorite_genre'].value_counts()

favorite_genre
Comedia        1112
Drama          1105
Documental     1095
Thriller       1090
Romance        1090
Acción         1082
Crime          1067
Action           22
COMEDIA          19
Crimen           18
CRIME            17
Romance          17
DRAMA            16
Documentary      16
Comedia          15
DOC              15
ACCIÓN           14
THRILLER         14
ROMANCE          14
comedy           12
Thriller         12
romance          12
drama            10
documental       10
accion            8
Drama             7
thriler           6
crime             5
Name: count, dtype: int64

anteriormente se utilizó describe() para poder evaluar algunos de los datos que teníamos, y encontramos un detalle: el valor más alto de  user_id es "17999"...

In [ ]:

ids_duplicados = df[df.duplicated(subset=['user_id'], keep=False)]
ids_duplicados.sort_values('user_id').head(10)

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
37,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8133,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8089,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
52,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
59,10059,39,Estándar,2976.6,Brasil,DRAMA,2024-06-22,1
8090,10059,39,Estándar,824.8,Brasil,Drama,2024-06-22,1
8010,10117,29,Estándar,713.9,Brasil,Documental,2020-12-20,1
117,10117,29,Estándar,713.9,Brasil,Documental,2020-12-20,1
8085,10128,19,Básico,638.7,Argentina,Drama,2020-06-17,1
128,10128,19,Básico,638.7,Argentina,Drama,2020-06-17,1
